<a href="https://colab.research.google.com/github/aymanr8-dot/ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026_10_16_walmart_warm_up_sku_mess_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
RAW_URL = 'https://raw.githubusercontent.com/jasontpv/DS2002FA26/main/notebooks/06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    try:
        # Colab opened from GitHub has no local data folder -> pull the real file from the repo
        cat = pd.read_csv(RAW_URL)
    except Exception:
        from io import StringIO
        cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
print(cat.shape)
cat.head()

(45, 7)


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# Case-insensitive filter on product_name containing 'pop'
pop_mask = cat['product_name'].str.lower().str.contains('pop', na=False)
pop = cat[pop_mask]
print('Rows that look like Pop-Tarts:', len(pop))
pop

Rows that look like Pop-Tarts: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
is_straw = pop['product_name'].str.lower().str.contains('strawberry', na=False)
straw = pop[is_straw]
not_straw = pop[~is_straw]

print('Strawberry Pop-Tart variants:', len(straw))
display(straw[['sku', 'product_name', 'category', 'package_size']])

print('Pop-Tart rows that are NOT strawberry (must stay separate):', len(not_straw))
display(not_straw[['sku', 'product_name']])

Strawberry Pop-Tart variants: 4


,sku,product_name,category,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,4 pack


Pop-Tart rows that are NOT strawberry (must stay separate): 0


,sku,product_name


**Pop-Tart rows that are a different product:** None. All four Pop-Tart rows (PT-12, SKU#459812, POPTART-STRAW, PT_STRAWBERRY) are strawberry, so all of them map to PT-12. If another flavor appeared, it would keep its own SKU.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
CANONICAL = 'PT-12'
sku_map = {s: CANONICAL for s in straw['sku'] if s != CANONICAL}
print(sku_map)

n_before = len(cat)
cat['canonical_sku'] = cat['sku'].map(sku_map).fillna(cat['sku'])

print('Distinct SKUs before:', cat['sku'].nunique(), '| after:', cat['canonical_sku'].nunique())
cat[['sku', 'canonical_sku', 'product_name']].head(8)

{'SKU#459812': 'PT-12', 'POPTART-STRAW': 'PT-12', 'PT_STRAWBERRY': 'PT-12'}
Distinct SKUs before: 45 | after: 42


,sku,canonical_sku,product_name
0,PT-12,PT-12,pop-tarts strawberry
1,SKU#459812,PT-12,Strawberry Pop-Tart
2,POPTART-STRAW,PT-12,Pop-Tarts Strawberry
3,PT_STRAWBERRY,PT-12,pop-tarts strawberry
4,WTR-24,WTR-24,BOTTLED WATER 24PK
5,WATER24PK,WATER24PK,Bottled Water 24-Pack
6,WTR_24PK,WTR_24PK,Water - 24 Pack
7,BATT-100,BATT-100,BATTERIES AA 4PK


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
print('BEFORE:')
print(cat['category'].value_counts(dropna=False).sort_index(), '\n')

cat['category_clean'] = cat['category'].str.strip().str.title()

print('AFTER strip + title-case:')
print(cat['category_clean'].value_counts(dropna=False).sort_index())

# Still two labels for one thing after normalizing case:
#   Breakfast  vs  Breakfast Foods
#   Emergency  vs  Emergency Supplies
#   Alcohol    vs  Alcoholic Beverages
#   Grocery    vs  Groceries
#   Baby       vs  Baby Products
print('\nDistinct categories before:', cat['category'].nunique(), '| after:', cat['category_clean'].nunique())

BEFORE:
category
ALCOHOL                1
Alcoholic Beverages    1
Baby                   2
Baby Products          1
Breakfast              2
Breakfast Foods        1
ELECTRONICS            2
EMERGENCY              3
ENTERTAINMENT          3
Electronics            1
Emergency              5
Emergency Supplies     3
GROCERY                3
Groceries              1
Grocery                1
HOUSEHOLD              1
Household              1
Snacks                 1
baby                   1
breakfast              1
electronics            1
emergency              1
entertainment          2
grocery                1
health                 2
household              2
snacks                 1
Name: count, dtype: int64 

AFTER strip + title-case:
category_clean
Alcohol                1
Alcoholic Beverages    1
Baby                   3
Baby Products          1
Breakfast              3
Breakfast Foods        1
Electronics            4
Emergency              9
Emergency Supplies     3
Entertainment 

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
import re

def name_key(name):
    """lowercase -> punctuation to spaces -> sort the words"""
    s = re.sub(r'[^a-z0-9\s]', ' ', str(name).lower())
    return ' '.join(sorted(s.split()))

cat['name_key'] = cat['product_name'].apply(name_key)

groups = (cat.groupby('name_key')
             .agg(n_skus=('sku', 'nunique'),
                  skus=('sku', list),
                  names=('product_name', list))
             .query('n_skus > 1')
             .sort_values('n_skus', ascending=False))

print(len(groups), 'candidate groups holding more than one SKU')
pd.set_option('display.max_colwidth', None)
groups

14 candidate groups holding more than one SKU


,n_skus,skus,names
name_key,,,
board game monopoly,3,"[BG-MONO, MONOPOLY-01, GAME-MONO]","[Monopoly Board Game, Monopoly Board Game, Board Game - Monopoly]"
pop strawberry tarts,3,"[PT-12, POPTART-STRAW, PT_STRAWBERRY]","[pop-tarts strawberry, Pop-Tarts Strawberry, pop-tarts strawberry]"
4 aa batteries pack,2,"[BAT-AA4, BATT_AA_4PK]","[AA Batteries 4-Pack, AA Batteries 4-Pack]"
aid first kit,2,"[FAID-01, FIRSTAID-KIT]","[First-Aid Kit, First Aid Kit]"
bread white,2,"[BREAD-WH, WHITEBREAD]","[WHITE BREAD, WHITE BREAD]"
24 beer light pack,2,"[BEER-24, LTBEER24]","[Light Beer (24-pack), Light Beer (24-pack)]"
butter peanut,2,"[PB-16, PNUTBTR-16]","[PEANUT BUTTER, Peanut Butter]"
candles emergency,2,"[CNDL-EM, CANDLE-EMRG]","[Candles (Emergency), Emergency Candles]"
cards playing,2,"[CARDS-01, PLAYCARD-STD]","[Playing Cards, PLAYING CARDS]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
rows_before = n_before
assert cat['canonical_sku'].notna().all(), 'mapping produced nulls in canonical_sku'
assert len(cat) == rows_before, 'row count changed after mapping'

# every unmapped SKU should be untouched
assert (cat.loc[~cat['sku'].isin(sku_map), 'canonical_sku'] == cat.loc[~cat['sku'].isin(sku_map), 'sku']).all()

print(f'OK: {len(cat)} rows before and after, 0 nulls in canonical_sku, '
      f'{cat["sku"].nunique()} -> {cat["canonical_sku"].nunique()} distinct SKUs')

OK: 45 rows before and after, 0 nulls in canonical_sku, 45 -> 42 distinct SKUs


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Two other fragmented product families:

Bottled water 24-pack: WTR-24, WATER24PK and WTR_24PK. Q5 doesn’t catch these because “24PK”, “24-Pack” and “24 Pack” turn into different words.

AA batteries 4-pack: BATT-100, BAT-AA4 and BATT_AA_4PK. Q5 only grouped two of them, because BATT-100 says “4PK”.

I would group these by using the key in Q5, but swapping pk/pkg for "pack" and splitting numbers from units ("24pk" would become "24 pack"). Furthermore, I would confirm each match against subcategory before I map to one canonical SKU. That fixes the batteries, but "Water - 24 Pack" still wouldn't match because it drops the word "bottled", so it needs a looser check (e.g. token overlap or fuzzy matching).

b) Consolidation stops being a cleaning task when rows differ in attributes that affect pricing, inventory or ordering and not just spelling. Pack size, supplier and cost are the attributes. My example from this catalog would be the strawberry Pop-Tarts. They come from 4 different suppliers and range in prices (\$1.93–\$2.26), with some being 6-packs and others being 4-packs, thus, merging them could mix up per-unit costs and reorder quantities. I would ask the Breakfast category manager or merchandising and on the other hand, procurement, for the supplier question.